In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
# Pull the original open-source SRGAN code
!git clone https://github.com/Lornatang/SRGAN-PyTorch.git /kaggle/working/SRGAN-PyTorch

# Move into the folder
%cd /kaggle/working/SRGAN-PyTorch

# Delete any line containing 'torch' from the text file
!sed -i '/torch/d' requirements.txt

# Install the rest of the libraries safely!
!pip install -r requirements.txt

fatal: destination path '/kaggle/working/SRGAN-PyTorch' already exists and is not an empty directory.
/kaggle/working/SRGAN-PyTorch
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=6c73dcf17c3824cd02c901bbe188c72a5c3547cd4598367428b951d205692bc5
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing


In [6]:
!find /kaggle/input -maxdepth 3

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/maryclauds
/kaggle/input/datasets/maryclauds/sar-image-patches


In [7]:
# Find the exact path to your 90-epoch brain
!find /kaggle/input -name "epoch_90.pth.tar"

# Find the exact path to your High-Res images
!find /kaggle/input -type d -name "train_HR"

/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar
/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset/train_HR


In [8]:
import yaml

# Open the original YAML file
with open('./configs/train/SRGAN_x4-ImageNet.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# 1. Fix the Hardware and Compiler
cfg['DEVICE_ID'] = 0
cfg['MODEL']['EMA']['COMPILED'] = False
cfg['MODEL']['G']['COMPILED'] = False
cfg['MODEL']['D']['COMPILED'] = False

# 2. Map your Image Folders
base_path = "/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset"
cfg['TRAIN']['DATASET']['TRAIN_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 3. Link your 90-Epoch Brain
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = "/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar"

# 4. Set the Save Directory so you don't lose data
cfg['TEST']['SAVE_IMAGE_DIR'] = "/kaggle/working/samples/images/"

# Save the updated file back to the folder
with open('./configs/train/SRGAN_x4-ImageNet.yaml', 'w') as f:
    yaml.dump(cfg, f)

print("✅ Phase 2 Configuration successfully updated for Kaggle!")

✅ Phase 2 Configuration successfully updated for Kaggle!


In [9]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-16 14:07:05.379002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773670025.570840     166 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773670025.631460     166 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773670026.129852     166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773670026.129903     166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773670026.129907     166 computation_placer.cc:177] computation placer alr